# Missão Aurora - Verificação pré-decolagem

Atividade Integradora, Fase 1 (FIAP)

**Integrantes:** Murilo Ribeiro Falconeri, Gabriel Ferreira da Silva, Gustavo Rocha Caxias, José Kauan Medeiros Machado

No Colab é só usar **Ambiente de execução > Executar tudo**. A parte de IA precisa da chave `GEMINI_API_KEY` cadastrada em Secrets; sem ela, só essa parte é pulada.

In [ ]:
%pip install -q "google-genai>=2.0,<3.0"

## 1.1 Telemetria

São seis leituras feitas antes da decolagem: temperatura interna e externa, integridade estrutural (0 ou 1), nível de energia, pressão do tanque e status dos módulos críticos. Cada uma é classificada como ok, alerta ou crítico:

| Parâmetro | Ok | Alerta | Crítico |
|---|---|---|---|
| Temp. interna | -40 a 50 °C | 50 a 70 °C | > 70 °C ou < -40 °C |
| Temp. externa | -50 a 60 °C | 60 a 80 °C ou -60 a -50 °C | > 80 °C ou < -60 °C |
| Integridade | 1 | - | 0 |
| Energia | ≥ 95% | 40 a 94% | < 40% |
| Pressão | 4.0 a 5.5 bar | 3.5 a 3.9 ou 5.6 a 6.5 bar | < 3.5 ou > 6.5 bar |
| Módulos | ok | - | falha |

## 1.2 Algoritmo

Se qualquer parâmetro estiver crítico ou em alerta, a decolagem é abortada. Só com tudo ok o resultado é PRONTO PARA DECOLAR. O fluxograma está no README do repositório.

## 1.3 Script em Python

In [1]:
USE_COLORS = True


BATTERY_CAPACITY_KWH = 14.4


LAUNCH_CONSUMPTION_KWH = 3.0


ENERGY_LOSS_RATE = 0.10


CRUISE_CONSUMPTION_KW = 1.2


RESERVE_RATE = 0.20


class Colors:
    RESET = '\033[0m'
    RED = '\033[91m'
    GREEN = '\033[92m'
    YELLOW = '\033[93m'
    CYAN = '\033[96m'


def colorize(text, color):
    if USE_COLORS:
        return f"{color}{text}{Colors.RESET}"
    return text


USE_COLORS = False

In [2]:
def get_float_input(prompt):
    while True:
        try:
            return float(input(prompt))
        except ValueError:
            print("Erro: Digite um valor numérico válido.")


def get_percentage_input(prompt):
    while True:
        value = get_float_input(prompt)
        if 0 <= value <= 100:
            return value
        print("Erro: O nível de energia deve estar entre 0 e 100%.")


def get_status_input(prompt):
    while True:
        status = input(prompt).lower().strip()
        if status in ["ok", "falha", "operacional", "critico"]:
            return status
        print("Erro: Digite 'ok', 'falha', 'operacional' ou 'critico'.")


def get_integrity_input(prompt):
    while True:
        try:
            value = int(input(prompt))
            if value in [0, 1]:
                return value
            print("Erro: Digite 0 (comprometida) ou 1 (íntegra).")
        except ValueError:
            print("Erro: Digite 0 ou 1.")


def get_system_inputs():
    internal_temperature = get_float_input("Digite a temperatura interna (°C): ")
    external_temperature = get_float_input("Digite a temperatura externa (°C): ")
    structural_integrity = get_integrity_input("Digite a integridade estrutural (0/1): ")
    energy_level = get_percentage_input("Digite o nível de energia (%): ")
    pressure_value = get_float_input("Digite a pressão do tanque (bar): ")
    module_status = get_status_input("Digite o status dos módulos críticos (ok/falha): ")
    return internal_temperature, external_temperature, structural_integrity, energy_level, pressure_value, module_status

In [3]:
def calculate_internal_temperature(internal):
    if internal > 70:
        return "Temperatura interna: CRÍTICA", "critico"
    elif 50 <= internal <= 70:
        return "Temperatura interna: ALERTA", "alerta"
    elif 20 <= internal <= 25:
        return "Temperatura interna: IDEAL", "ok"
    elif -40 <= internal <= 50:
        return "Temperatura interna: OPERACIONAL", "ok"
    else:
        return "Temperatura interna: FORA DA FAIXA", "critico"


def calculate_external_temperature(external):
    if external < -60 or external > 80:
        return "Temperatura externa: CRÍTICA", "critico"
    elif external < -50 or external > 60:
        return "Temperatura externa: ALERTA", "alerta"
    elif 0 <= external <= 30:
        return "Temperatura externa: IDEAL", "ok"
    elif -50 <= external <= 60:
        return "Temperatura externa: OPERACIONAL", "ok"
    else:
        return "Temperatura externa: FORA DA FAIXA", "critico"


def calculate_energy_level(energy):
    if energy < 40:
        return "Energia: CRÍTICA", "critico"
    elif 40 <= energy < 70:
        return "Energia: ALERTA", "alerta"
    elif energy < 95:
        return "Energia: ABAIXO DO MÍNIMO PARA DECOLAGEM", "alerta"
    else:
        return "Energia: PRONTA PARA DECOLAGEM", "ok"


def calculate_pressure(pressure_val):
    if pressure_val < 3.5 or pressure_val > 6.5:
        return "Pressão: CRÍTICA", "critico"
    elif (3.5 <= pressure_val <= 3.9) or (5.6 <= pressure_val <= 6.0):
        return "Pressão: ALERTA", "alerta"
    elif 4.0 <= pressure_val <= 5.5:
        return "Pressão: OPERACIONAL", "ok"
    else:
        return "Pressão: ALERTA", "alerta"


def check_structural_integrity(integrity):
    if integrity == 1:
        return "Integridade estrutural: ÍNTEGRA", "ok"
    else:
        return "Integridade estrutural: COMPROMETIDA", "critico"


def check_module_status(status):
    if status.lower() in ["ok", "operacional"]:
        return "Módulos: FUNCIONANDO", "ok"
    else:
        return "Módulos: FALHA", "critico"


def calculate_energy_autonomy(energy):
    stored_energy = BATTERY_CAPACITY_KWH * (energy / 100)
    energy_losses = stored_energy * ENERGY_LOSS_RATE
    usable_energy = stored_energy - energy_losses
    reserve_energy = BATTERY_CAPACITY_KWH * RESERVE_RATE
    remaining_energy = usable_energy - LAUNCH_CONSUMPTION_KWH - reserve_energy

    if remaining_energy <= 0:
        autonomy_hours = 0.0
    else:
        autonomy_hours = remaining_energy / CRUISE_CONSUMPTION_KW

    return stored_energy, energy_losses, usable_energy, reserve_energy, remaining_energy, autonomy_hours


def print_energy_analysis(energy):
    stored, losses, usable, reserve, remaining, hours = calculate_energy_autonomy(energy)

    print("\n=== ANÁLISE ENERGÉTICA ===")
    print(f"Capacidade total do banco: {BATTERY_CAPACITY_KWH:.1f} kWh")
    print(f"Carga atual: {energy:.1f}%")
    print(f"Energia armazenada: {stored:.1f} kWh")
    print(f"Perdas energéticas ({ENERGY_LOSS_RATE * 100:.0f}%): -{losses:.1f} kWh")
    print(f"Energia útil: {usable:.1f} kWh")
    print(f"Consumo na decolagem: -{LAUNCH_CONSUMPTION_KWH:.1f} kWh")
    print(f"Reserva operacional ({RESERVE_RATE * 100:.0f}%): -{reserve:.1f} kWh")
    print(f"Energia disponível após a decolagem: {remaining:.1f} kWh")

    if hours <= 0:
        print(colorize("Autonomia estimada: INSUFICIENTE PARA A MISSÃO", Colors.RED))
    else:
        print(colorize(f"Autonomia estimada: {hours:.1f} horas", Colors.CYAN))

    return hours


def verify_launch(internal_temp, external_temp, integrity, energy, pressure_val, module_status):
    results = []
    
    msg_temp, status_temp = calculate_internal_temperature(internal_temp)
    results.append((msg_temp, status_temp))
    
    msg_ext_temp, status_ext_temp = calculate_external_temperature(external_temp)
    results.append((msg_ext_temp, status_ext_temp))
    
    msg_integrity, status_integrity = check_structural_integrity(integrity)
    results.append((msg_integrity, status_integrity))
    
    msg_energy, status_energy = calculate_energy_level(energy)
    results.append((msg_energy, status_energy))
    
    msg_pressure, status_pressure = calculate_pressure(pressure_val)
    results.append((msg_pressure, status_pressure))
    
    msg_modules, status_modules = check_module_status(module_status)
    results.append((msg_modules, status_modules))
    
    print("\n=== DIAGNÓSTICO DO SISTEMA ===")
    for msg, status in results:
        if status == "critico":
            print(colorize(msg, Colors.RED))
        elif status == "alerta":
            print(colorize(msg, Colors.YELLOW))
        else:
            print(colorize(msg, Colors.GREEN))
    
    has_critical = any(status == "critico" for _, status in results)
    has_alert = any(status == "alerta" for _, status in results)
    
    print("\n=== RESULTADO FINAL ===")
    if has_critical:
        result_msg = "DECOLAGEM ABORTADA - Sistema em estado crítico"
        print(colorize(result_msg, Colors.RED))
        return False
    elif has_alert:
        result_msg = "DECOLAGEM ABORTADA - Alertas detectados nos sistemas"
        print(colorize(result_msg, Colors.RED))
        return False
    else:
        result_msg = "PRONTO PARA DECOLAR - Todos os sistemas operacionais"
        print(colorize(result_msg, Colors.GREEN))
        return True

In [4]:
SCENARIOS = [
    {"nome": "Tudo ok", "temperatura_interna": 22, "temperatura_externa": 20, "integridade": 1, "energia": 98, "pressao": 4.5, "modulos": "ok"},
    {"nome": "Pressão do tanque em alerta", "temperatura_interna": 22, "temperatura_externa": 20, "integridade": 1, "energia": 98, "pressao": 3.7, "modulos": "ok"},
    {"nome": "Estrutura comprometida", "temperatura_interna": 22, "temperatura_externa": 20, "integridade": 0, "energia": 98, "pressao": 4.5, "modulos": "ok"},
    {"nome": "Várias falhas", "temperatura_interna": 65, "temperatura_externa": 70, "integridade": 1, "energia": 35, "pressao": 6.8, "modulos": "falha"},
]


def run_scenario(scenario, use_ai=False):
    print(f"\n>>> {scenario['nome']}")
    launch_ok = verify_launch(
        scenario["temperatura_interna"],
        scenario["temperatura_externa"],
        scenario["integridade"],
        scenario["energia"],
        scenario["pressao"],
        scenario["modulos"],
    )
    autonomy_hours = print_energy_analysis(scenario["energia"])

    if use_ai:
        telemetry = {
            "temperatura interna": f"{scenario['temperatura_interna']} °C",
            "temperatura externa": f"{scenario['temperatura_externa']} °C",
            "integridade estrutural": scenario["integridade"],
            "nível de energia": f"{scenario['energia']}%",
            "pressão do tanque": f"{scenario['pressao']} bar",
            "status dos módulos": scenario["modulos"],
        }
        run_ai_analysis(telemetry, launch_ok, autonomy_hours)

In [5]:
for scenario in SCENARIOS[:3]:
    run_scenario(scenario)


>>> Tudo ok

=== DIAGNÓSTICO DO SISTEMA ===
Temperatura interna: IDEAL
Temperatura externa: IDEAL
Integridade estrutural: ÍNTEGRA
Energia: PRONTA PARA DECOLAGEM
Pressão: OPERACIONAL
Módulos: FUNCIONANDO

=== RESULTADO FINAL ===
PRONTO PARA DECOLAR - Todos os sistemas operacionais

=== ANÁLISE ENERGÉTICA ===
Capacidade total do banco: 14.4 kWh
Carga atual: 98.0%
Energia armazenada: 14.1 kWh
Perdas energéticas (10%): -1.4 kWh
Energia útil: 12.7 kWh
Consumo na decolagem: -3.0 kWh
Reserva operacional (20%): -2.9 kWh
Energia disponível após a decolagem: 6.8 kWh
Autonomia estimada: 5.7 horas

>>> Pressão do tanque em alerta

=== DIAGNÓSTICO DO SISTEMA ===
Temperatura interna: IDEAL
Temperatura externa: IDEAL
Integridade estrutural: ÍNTEGRA
Energia: PRONTA PARA DECOLAGEM
Pressão: ALERTA
Módulos: FUNCIONANDO

=== RESULTADO FINAL ===
DECOLAGEM ABORTADA - Alertas detectados nos sistemas

=== ANÁLISE ENERGÉTICA ===
Capacidade total do banco: 14.4 kWh
Carga atual: 98.0%
Energia armazenada: 14.1 k

In [6]:
# para digitar os valores na mão:
# valores = get_system_inputs()
# verify_launch(*valores)
# print_energy_analysis(valores[3])

## 1.4 Análise energética

Bateria de 14.4 kWh (4 baterias de 3.6 kWh, como na Orion da NASA), 3.0 kWh gastos na decolagem, 10% de perdas, 20% de reserva e consumo de 1.2 kW depois do lançamento.

Com 98% de carga:

- armazenada: 14.4 x 0.98 = 14.11 kWh
- útil, tirando as perdas: 14.11 - 1.41 = 12.70 kWh
- disponível, tirando decolagem e reserva: 12.70 - 3.0 - 2.88 = 6.82 kWh
- autonomia: 6.82 / 1.2 = 5.7 horas

In [7]:
for charge in [100, 95, 90, 70, 50, 45]:
    remaining, hours = calculate_energy_autonomy(charge)[4:]
    print(f"{charge}%: {remaining:.2f} kWh disponíveis, " + (f"{hours:.1f} h" if hours > 0 else "insuficiente"))

100%: 7.08 kWh disponíveis, 5.9 h
95%: 6.43 kWh disponíveis, 5.4 h
90%: 5.78 kWh disponíveis, 4.8 h
70%: 3.19 kWh disponíveis, 2.7 h
50%: 0.60 kWh disponíveis, 0.5 h
45%: -0.05 kWh disponíveis, insuficiente


## 1.5 Análise assistida por IA

A telemetria, o resultado e a autonomia vão para o Gemini (`gemini-3.5-flash-lite`), que devolve a classificação dos dados, anomalias e riscos. A IA não muda a decisão do algoritmo. Mesmo código do `fase1/analise_ia.py`, só que aqui a chave vem dos Secrets do Colab.

In [8]:
import os

MODELS = ["gemini-3.5-flash-lite", "gemini-3.1-flash-lite"]


SYSTEM_PROMPT = """Você analisa a telemetria de pré-decolagem de um foguete.

Faixas seguras:
- temperatura interna: -40 a 50 °C (alerta até 70)
- temperatura externa: -50 a 60 °C (alerta até -60 e 80)
- energia: mínimo de 95% para decolar (crítico abaixo de 40%)
- pressão do tanque: 4.0 a 5.5 bar (alerta de 3.5 a 6.5)
- integridade estrutural: 1 ok, 0 comprometida
- módulos críticos: ok ou operacional

Responda em português, curto e sem markdown, com:
1. Classificação de cada dado (normal, atenção ou crítico)
2. Possíveis anomalias
3. Sugestões de risco

Use só os dados recebidos. A decisão de decolar é do algoritmo, não sua."""


def run_ai_analysis(telemetry, launch_ok, autonomy_hours):
    api_key = load_api_key()
    if not api_key:
        print("\nAnálise por IA ignorada: GEMINI_API_KEY não encontrada no .env")
        return

    try:
        from google import genai
        from google.genai import types
    except ImportError:
        print("\nAnálise por IA ignorada.")
        return

    verdict = "PRONTO PARA DECOLAR" if launch_ok else "DECOLAGEM ABORTADA"
    data = "\n".join(f"- {name}: {value}" for name, value in telemetry.items())
    prompt = f"Telemetria:\n{data}\nVeredito do algoritmo: {verdict}\nAutonomia estimada: {autonomy_hours:.1f} horas"

    client = genai.Client(api_key=api_key, http_options=types.HttpOptions(timeout=30000))
    config = types.GenerateContentConfig(
        system_instruction=SYSTEM_PROMPT,
        temperature=0.2,
        thinking_config=types.ThinkingConfig(thinking_level="minimal"),
        automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
    )

    print("\nConsultando a IA...")
    response = None
    for model in MODELS:
        try:
            response = client.models.generate_content(model=model, contents=prompt, config=config)
            break
        except Exception as error:
            last_error = getattr(error, "message", error)

    if response is None:
        print("Não foi possível consultar a IA:", last_error)
        return

    print("\nAnálise da IA:")
    print((response.text or "A IA não retornou resposta.").strip())
    print()


def load_api_key():
    key = os.getenv("GEMINI_API_KEY")
    if key:
        return key
    try:
        from google.colab import userdata
        return userdata.get("GEMINI_API_KEY")
    except Exception:
        return None

In [9]:
run_scenario(SCENARIOS[3], use_ai=True)


>>> Várias falhas

=== DIAGNÓSTICO DO SISTEMA ===
Temperatura interna: ALERTA
Temperatura externa: ALERTA
Integridade estrutural: ÍNTEGRA
Energia: CRÍTICA
Pressão: CRÍTICA
Módulos: FALHA

=== RESULTADO FINAL ===
DECOLAGEM ABORTADA - Sistema em estado crítico

=== ANÁLISE ENERGÉTICA ===
Capacidade total do banco: 14.4 kWh
Carga atual: 35.0%
Energia armazenada: 5.0 kWh
Perdas energéticas (10%): -0.5 kWh
Energia útil: 4.5 kWh
Consumo na decolagem: -3.0 kWh
Reserva operacional (20%): -2.9 kWh
Energia disponível após a decolagem: -1.3 kWh
Autonomia estimada: INSUFICIENTE PARA A MISSÃO

Consultando a IA...

Análise da IA:
1. Classificações: temperatura interna atenção, temperatura externa normal, integridade estrutural normal, nível de energia crítico, pressão do tanque crítico, status dos módulos crítico.
2. Possíveis anomalias: superaquecimento interno, energia abaixo do mínimo para decolagem, sobrepressão no tanque e falha nos módulos críticos.
3. Sugestões de risco: risco iminente de pe

Em alguns testes a IA disse que a temperatura externa de 70 °C era normal, mas pela tabela ela está em alerta. É por isso que a decisão fica com o algoritmo.

## 1.6 Reflexão crítica

Construir o sistema de verificação da Aurora nos fez perceber que, por trás de cada faixa numérica que definimos, existe uma escolha com peso ético. Decidir que a energia precisa estar acima de 95% para liberar a decolagem, ou que qualquer falha de integridade estrutural já aborta a missão, não é só uma questão técnica: é assumir que, na dúvida, é melhor abortar do que arriscar. Por isso o algoritmo foi pensado para ser conservador, e por isso a decisão final continua sendo dele, e não da IA. A análise por inteligência artificial (item 1.5) entra como uma segunda opinião, algo que ajuda a interpretar os dados, mas nunca substitui um critério determinístico e auditável quando o que está em jogo é uma decisão crítica.

Esse cuidado técnico também nos fez pensar num quadro mais amplo: por que investir em telemetria, automação e engenharia de sistemas críticos, se estamos falando de uma missão fictícia? Porque essas mesmas ferramentas, na vida real, vão muito além de foguetes e aparecem em monitoramento ambiental, em resposta a desastres e em telecomunicações. Ao mesmo tempo, não dá para ignorar que programas espaciais de verdade consomem recursos enormes, o que levanta uma pergunta legítima: até onde vale a pena investir em exploração espacial quando existem tantas necessidades mais urgentes na Terra? Não é uma pergunta com resposta fácil, mas é uma que qualquer projeto nessa área deveria carregar junto.

Por fim, a própria análise energética do projeto (item 1.4) acabou virando, sem planejarmos assim, um pequeno exercício de sustentabilidade: calcular autonomia, prever perdas e definir uma reserva mínima de segurança é exatamente o tipo de raciocínio que sistemas sustentáveis precisam ter, sejam eles espaciais ou não. Pensar em eficiência e desperdício desde o design, e não como correção depois do fato, foi talvez o maior aprendizado prático que tiramos da Aurora.